# Flight Price Prediction - Simplified Pipeline
This notebook trains the two top-performing models (**Random Forest Regressor** and **Hist Gradient Boosting Regressor**) for flight price prediction using a target log-transformation to achieve high $R^2$ performance ($R^2 > 0.85$ on log-price scale, high variance explained on original scale).

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import kagglehub

SEED = 42
DATA_DIR = "."

# 1. Load Dataset
# dataset_path = kagglehub.competition_download('flight-price-prediction')
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
# train_df = pd.read_csv(os.path.join(dataset_path, 'train.csv'))
# test_df = pd.read_csv(os.path.join(dataset_path, 'test.csv'))
sub_sample = pd.read_csv(f"{DATA_DIR}/submission.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (18076, 12)
Test shape: (4520, 11)


In [3]:
# 2. Feature Engineering & Preprocessing
df = train_df[train_df.price > 0].copy()
test = test_df.copy()

def make_features(d):
    d = d.copy()
    d['route'] = d.source_city + '_' + d.destination_city
    d['stops_n'] = d.stops.map({'zero': 0, 'one': 1, 'two_or_more': 2})
    d['near'] = (d.days_left <= 14).astype(int)
    return d

df = make_features(df)
test_f = make_features(test)

cat_cols = ['airline', 'source_city', 'destination_city', 'departure_time',
            'arrival_time', 'stops', 'flight_class', 'route']

all_X = pd.concat([df.drop(columns=['id', 'price', 'flight']),
                   test_f.drop(columns=['id', 'flight'])], keys=['tr', 'te'])
all_X = pd.get_dummies(all_X, columns=cat_cols)
all_X['flight'] = pd.Categorical(pd.concat([df.flight, test_f.flight])).codes

X = all_X.loc['tr']
X_test = all_X.loc['te']
y = df.price.values
y_log = np.log1p(y)

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (18051, 57)


In [4]:
# 3. Train-Test Split (80% Train, 20% Validation)
X_train, X_val, y_train_log, y_val_log = train_test_split(X, y_log, test_size=0.2, random_state=SEED)
y_val_true = np.expm1(y_val_log)

# Model 1: Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=30, max_features=0.7, n_jobs=-1, random_state=SEED
)
rf_model.fit(X_train, y_train_log)
rf_preds_log = rf_model.predict(X_val)
rf_preds = np.expm1(rf_preds_log)

print("=== Model 1: Random Forest Regressor ===")
print(f"R2 Score (log scale)  : {r2_score(y_val_log, rf_preds_log):.4f}")
print(f"R2 Score (real scale) : {r2_score(y_val_true, rf_preds):.4f}")
print(f"MAE (rupees)          : {mean_absolute_error(y_val_true, rf_preds):.2f}")

=== Model 1: Random Forest Regressor ===
R2 Score (log scale)  : 0.8811
R2 Score (real scale) : 0.7761
MAE (rupees)          : 38.64


In [5]:
# Model 2: Hist Gradient Boosting Regressor
hgb_model = HistGradientBoostingRegressor(
    max_iter=500, learning_rate=0.08, max_leaf_nodes=63, random_state=SEED
)
hgb_model.fit(X_train, y_train_log)
hgb_preds_log = hgb_model.predict(X_val)
hgb_preds = np.expm1(hgb_preds_log)

print("=== Model 2: Hist Gradient Boosting Regressor ===")
print(f"R2 Score (log scale)  : {r2_score(y_val_log, hgb_preds_log):.4f}")
print(f"R2 Score (real scale) : {r2_score(y_val_true, hgb_preds):.4f}")
print(f"MAE (rupees)          : {mean_absolute_error(y_val_true, hgb_preds):.2f}")

=== Model 2: Hist Gradient Boosting Regressor ===
R2 Score (log scale)  : 0.8984
R2 Score (real scale) : 0.8058
MAE (rupees)          : 39.26


In [6]:
# 4. Ensembling & Final Test Submission
blend_val_log = (rf_preds_log + hgb_preds_log) / 2
blend_val_preds = np.expm1(blend_val_log)

print("=== Ensemble Model (RF + HistGB) ===")
print(f"R2 Score (log scale)  : {r2_score(y_val_log, blend_val_log):.4f}")
print(f"R2 Score (real scale) : {r2_score(y_val_true, blend_val_preds):.4f}")

# Refit models on 100% of train data and generate test predictions
rf_model.fit(X, y_log)
hgb_model.fit(X, y_log)

final_test_log = (rf_model.predict(X_test) + hgb_model.predict(X_test)) / 2
sub = sub_sample.copy()
sub['price'] = np.expm1(final_test_log).round().astype(int)
sub.to_csv('submission_clean.csv', index=False)
print("Submission saved successfully!")

=== Ensemble Model (RF + HistGB) ===
R2 Score (log scale)  : 0.8961
R2 Score (real scale) : 0.8001
Submission saved successfully!
